# 4 · A zoo of finite element spaces 🔬

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=04-fespaces.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/04-fespaces.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>


Every space we have used — `H1`, `L2`, `VectorH1` — is a set of **basis (shape)
functions**, one per degree of freedom. A `GridFunction` is just a list of
coefficients in front of those functions. A wonderful way to *understand* a
space is to **draw a single shape function**: set one coefficient to `1`, the
rest to `0`, and look.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.4))

def shape_functions(space, dofs):
    """A multidim GridFunction holding one basis function per requested dof."""
    gf = GridFunction(space, multidim=len(dofs))
    for i, d in enumerate(dofs):
        gf.vecs[i][:] = 0
        gf.vecs[i][d] = 1                  # activate a single basis function
    return gf

## 1. `H1` — continuous shape functions

The Lagrange-type `H1` shape functions are **continuous** across element edges:
their graphs join up with no jumps. That is exactly what makes `H1` the right
home for second-order problems like Poisson and heat. Press play to flip through
a few of them.

In [ ]:
fesH1 = H1(mesh, order=3)
gf = shape_functions(fesH1, [10, 18, 25, 33])
Draw(gf, mesh, "H1 shape fn", interpolate_multidim=False, animate=True,
     deformation=True)

## 2. `L2` — discontinuous shape functions

`L2` functions live **independently on each element** — neighbouring elements
share nothing, so a shape function is a bump confined to a single triangle. This
locality is what the discontinuous-Galerkin transport scheme will exploit
(notebook 14), and it gives the block-diagonal mass matrix.

In [ ]:
fesL2 = L2(mesh, order=3)
gf = shape_functions(fesL2, [12, 20, 28, 36])
Draw(gf, mesh, "L2 shape fn", interpolate_multidim=False, animate=True,
     deformation=True)

## 3. `HDiv` and `HCurl` — vector-valued, partially continuous

Not every space is scalar. `HDiv` shape functions are **vector fields** whose
*normal* component is continuous across edges (ideal for fluxes / flow);
`HCurl` keeps the *tangential* component continuous (ideal for electromagnetics).
Drawn as arrows, you can see the field is well-defined across an edge in one
direction but may jump in the other.

In [ ]:
fesHDiv = HDiv(mesh, order=2)
gf = shape_functions(fesHDiv, [15, 30, 45, 55])
Draw(gf, mesh, "HDiv shape fn", interpolate_multidim=False, animate=True)

## 4. How dofs are classified

Internally NGSolve labels every dof by how it **couples** between elements —
local (interior), interface (shared on edges/faces) or wirebasket (the coarse
skeleton used by `bddc`). This is the very information the solvers in notebook 11
exploit. We can simply ask the space.

In [ ]:
from collections import Counter
fes = H1(mesh, order=3)
kinds = Counter(str(fes.CouplingType(i)).split(".")[-1] for i in range(fes.ndof))
print(f"H1 order 3 — {fes.ndof} dofs:")
for kind, count in kinds.items():
    print(f"  {kind:16s}: {count}")

:::{dropdown} 🧠 Quiz — which space would you pick, and why?
* **Smooth scalar field** (temperature, potential) → `H1`: continuity is built in.
* **Transport / conservation laws, sharp fronts** → `L2`: locality + a cheap,
  block-diagonal mass matrix, and room for upwind fluxes.
* **Flux- or flow-type vector fields** (`u` with `div u = 0`) → `HDiv`: the
  normal component is continuous, so mass is conserved across edges.
* **Electromagnetic fields** → `HCurl`: the tangential component is continuous.

Matching the space to the *physics* — exactly which quantity must be continuous —
is one of the quiet superpowers of the finite element method. NGSolve gives you
all of them through the same `TnT()` / `BilinearForm` interface you already know.
:::

Next: the fifth deed up close — how the linear system is actually **solved**
(free dofs, lifting boundary data, static condensation).